# inf-masking — worked example 2: Pad-mask attention dropping PAD keys

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inf-masking`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A per-batch key padding mask `(B, S_k)` is broadcast to `(B, 1, S_k)` and applied with `masked_fill(-inf)` to `(B, S_q, S_k)` scores, so every query row in a batch element drops the same PAD keys. After softmax the PAD columns carry exactly zero attention mass.

## Worked solution

We mask padding tokens out of attention.

1. `scores` is `(B, S_q, S_k)`; `pad_mask` is `(B, S_k)` with `True` marking PAD keys.
2. Insert a query axis: `pad_mask.unsqueeze(1)` becomes `(B, 1, S_k)`, which broadcasts across all `S_q` query rows but stays independent per batch element.
3. `scores.masked_fill(that, float('-inf'))` sets every (query, PAD-key) entry to `-inf`.
4. `softmax(dim=-1)` over the key axis gives PAD columns exactly zero weight, and the non-PAD weights renormalize to 1 per row.
5. We verify the masked columns are zero for a batch element and that rows sum to 1.

In [ ]:
import torch as t

t.manual_seed(1)
scores = t.randn(2, 3, 4)
pad_mask = t.tensor([[False, False, True, True], [False, True, False, False]])

def pad_mask_attention(scores, pad_mask):
    masked = scores.masked_fill(pad_mask.unsqueeze(1), float('-inf'))
    return masked.softmax(dim=-1)

w = pad_mask_attention(scores, pad_mask)
print('batch0 PAD cols zero:', bool((w[0, :, 2:] == 0).all()))
print('rows sum to 1:', bool(t.allclose(w.sum(-1), t.ones(2, 3))))